In [1]:
import pandas as pd
import numpy as np
import joblib
import sys
from pathlib import Path

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*ntree_limit is deprecated.*"
)

print("Deprecation warning suppressed.")

Deprecation warning suppressed.


In [3]:
PROJECT_ROOT = Path("..").resolve()

print(PROJECT_ROOT)

C:\Users\Lum-neh Angela\Documents\Data\CreditScoring_Kaggle


In [4]:
print("Project folders:\n")

for item in PROJECT_ROOT.iterdir():
    print(item.name)

Project folders:

.gitignore
.ipynb_checkpoints
assets
data
images
models
notebooks
README.md
reports
requirements.txt
src


In [5]:
MODELS_DIR = PROJECT_ROOT / "models"

print("Models folder:")
print(MODELS_DIR)

print("\nFiles:")
for file in MODELS_DIR.iterdir():
    print(" -", file.name)

Models folder:
C:\Users\Lum-neh Angela\Documents\Data\CreditScoring_Kaggle\models

Files:
 - preprocessor.pkl
 - xgboost_credit_score.pkl


In [6]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed data folder:")
print(PROCESSED_DIR)

print("\nFiles:")
for file in PROCESSED_DIR.iterdir():
    print(" -", file.name)

Processed data folder:
C:\Users\Lum-neh Angela\Documents\Data\CreditScoring_Kaggle\data\processed

Files:
 - clean_df.pkl
 - feature_names.pkl
 - monitoring_reference.pkl
 - test_customer_ids.pkl
 - test_df.pkl
 - train_df.pkl
 - val_df.pkl
 - X_test_processed.pkl
 - X_train_processed.pkl
 - X_val_processed.pkl
 - y_test.pkl
 - y_train.pkl
 - y_val.pkl


In [7]:
SRC_DIR = PROJECT_ROOT / "src"

print("Source folder:")
print(SRC_DIR)

print("\nFiles:")
for file in SRC_DIR.iterdir():
    print(" -", file.name)

Source folder:
C:\Users\Lum-neh Angela\Documents\Data\CreditScoring_Kaggle\src

Files:
 - .ipynb_checkpoints
 - api.py
 - explainability.py
 - monitoring.py
 - preprocessing.py
 - __pycache__


In [8]:
# Load the saved artifacts
model = joblib.load(MODELS_DIR / "xgboost_credit_score.pkl")

preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")

feature_names = joblib.load(PROCESSED_DIR / "feature_names.pkl")

C:\Users\Lum-neh Angela\anaconda3\lib\site-packages\xgboost\compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [9]:
# Verify
print("Model:", type(model))
print("Preprocessor:", type(preprocessor))
print("Number of features:", len(feature_names))

Model: <class 'xgboost.sklearn.XGBClassifier'>
Preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Number of features: 61


In [10]:
# Load test data
test_df = joblib.load(PROCESSED_DIR / "test_df.pkl")

X_test_processed = joblib.load(PROCESSED_DIR / "X_test_processed.pkl")

y_test = joblib.load(PROCESSED_DIR / "y_test.pkl")

In [11]:
#Test load
print(type(X_test_processed))
print(X_test_processed.shape)

<class 'numpy.ndarray'>
(15000, 61)


In [12]:
print(type(y_test))
print(y_test.shape)

<class 'pandas.core.series.Series'>
(15000,)


In [13]:
#verify everything we currently have
print("Model:", type(model))
print("Preprocessor:", type(preprocessor))
print("Feature names:", len(feature_names))

print("\nTest data:")
print("test_df:", test_df.shape)
print("X_test_processed:", X_test_processed.shape)
print("y_test:", y_test.shape)

print("\nModel classes:")
print(model.classes_)

Model: <class 'xgboost.sklearn.XGBClassifier'>
Preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Feature names: 61

Test data:
test_df: (15000, 36)
X_test_processed: (15000, 61)
y_test: (15000,)

Model classes:
['Good' 'Poor' 'Standard']


In [14]:
# Verify the saved model
test_predictions = model.predict(X_test_processed)

test_probabilities = model.predict_proba(X_test_processed)

print("Predictions shape:", test_predictions.shape)
print("Probabilities shape:", test_probabilities.shape)

Predictions shape: (15000,)
Probabilities shape: (15000, 3)


In [15]:
# Verify the performance
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, test_predictions)

macro_f1 = f1_score(y_test, test_predictions, average="macro")

print("Test Accuracy:", round(accuracy, 4))
print("Test Macro F1:", round(macro_f1, 4))

Test Accuracy: 0.7079
Test Macro F1: 0.6858


In [16]:
# Test preprocessing + prediction from a raw customer record

sample_customer = test_df.iloc[[0]].copy()

print("Raw customer shape:", sample_customer.shape)
print("\nRaw customer data:")
display(sample_customer)

Raw customer shape: (1, 36)

Raw customer data:


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_History_Months,Num_Loan_Types,Has_auto_loan,Has_credit_builder_loan,Has_personal_loan,Has_home_equity_loan,Has_mortgage_loan,Has_student_loan,Has_debt_consolidation_loan,Has_payday_loan
80,0x167a,CUS_0xa66b,January,NaN,40.0,221-30-8554,Teacher,33751.27,2948.605833,5,...,230.0,3,1,1,1,0,0,0,0,0


#### Test Saved Deployment Pipeline

In [17]:
# Transform raw customer data using the saved preprocessor

sample_customer_processed = preprocessor.transform(sample_customer)

print("Processed data type:", type(sample_customer_processed))
print("Processed shape:", sample_customer_processed.shape)

Processed data type: <class 'numpy.ndarray'>
Processed shape: (1, 61)


In [18]:
# Generate prediction and probabilities

sample_prediction = model.predict(sample_customer_processed)[0]
sample_probabilities = model.predict_proba(sample_customer_processed)[0]

print("Predicted Credit Score:", sample_prediction)

print("\nPrediction probabilities:")
for class_name, probability in zip(model.classes_, sample_probabilities):
    print(f"{class_name}: {probability:.4f}")

Predicted Credit Score: Standard

Prediction probabilities:
Good: 0.0559
Poor: 0.1430
Standard: 0.8010


In [19]:
# Create a resuable deployment function
# Step 10: Create reusable prediction function

def predict_credit_score(customer_data):
    """
    Predict credit score from raw customer data.

    Parameters
    ----------
    customer_data : pandas.DataFrame
        Raw customer data with the same columns used during training.

    Returns
    -------
    dict
        Prediction and class probabilities.
    """
    
    # Transform raw data using the saved preprocessor
    processed_data = preprocessor.transform(customer_data)
    
    # Generate prediction
    prediction = model.predict(processed_data)[0]
    
    # Generate probabilities
    probabilities = model.predict_proba(processed_data)[0]
    
    return {
        "prediction": prediction,
        "probabilities": {
            class_name: float(probability)
            for class_name, probability in zip(model.classes_, probabilities)
        }
    }


# Test the function
result = predict_credit_score(sample_customer)

print("Prediction:", result["prediction"])
print("\nProbabilities:")

for class_name, probability in result["probabilities"].items():
    print(f"{class_name}: {probability:.4f}")

Prediction: Standard

Probabilities:
Good: 0.0559
Poor: 0.1430
Standard: 0.8010


In [20]:
# Load the explainability component

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.explainability import CreditScoreExplainer

print("CreditScoreExplainer loaded successfully.")

CreditScoreExplainer loaded successfully.


In [21]:
# Create the SHAP explainer

explainer = CreditScoreExplainer(
    model=model,
    feature_names=feature_names,
    numeric_features=preprocessor.transformers_[0][2],
    categorical_features=preprocessor.transformers_[1][2]
)

print("SHAP explainer created successfully.")

SHAP explainer created successfully.


In [22]:
# Generate SHAP explanation for the sample customer

sample_explanation = explainer.explain_observation(
    X_processed=sample_customer_processed,
    original_row=sample_customer,
    observation_index=0,
    top_n=10
)

print("Prediction:", sample_explanation["prediction"])
print("Probability:", round(sample_explanation["confidence"], 4))

print("\nClass probabilities:")
for class_name, probability in sample_explanation["probabilities"].items():
    print(f"{class_name}: {probability:.4f}")

print("\nTop contributing features:")
display(sample_explanation["explanation"])

Prediction: Standard
Probability: 0.801

Class probabilities:
Good: 0.0559
Poor: 0.1430
Standard: 0.8010

Top contributing features:


,Feature,Category,Value,SHAP,Abs_SHAP,Direction
28,Credit_Mix,Standard,Standard,0.446984,0.446984,Toward Standard
26,Month,January,January,0.230193,0.230193,Toward Standard
11,Outstanding_Debt,None,1328.93,-0.077405,0.077405,Away from Standard
5,Interest_Rate,None,20,0.073071,0.073071,Toward Standard
3,Num_Bank_Accounts,None,5,0.034796,0.034796,Toward Standard
7,Delay_from_due_date,None,16,-0.020983,0.020983,Away from Standard
30,Payment_Behaviour,High_spent_Medium_value_payments,High_spent_Medium_value_payments,0.020637,0.020637,Toward Standard
9,Changed_Credit_Limit,None,11.0,-0.017222,0.017222,Away from Standard
10,Num_Credit_Inquiries,None,4.0,0.015161,0.015161,Toward Standard
24,Has_debt_consolidation_loan,None,0,-0.015078,0.015078,Away from Standard


In [23]:
# Test deployment prediction on multiple customers

sample_customers = test_df.iloc[[0, 100, 200]].copy()

deployment_results = []

for index, row in sample_customers.iterrows():
    customer = row.to_frame().T
    
    result = predict_credit_score(customer)
    
    deployment_results.append({
        "Customer_ID": row["Customer_ID"],
        "Actual": y_test.loc[index],
        "Predicted": result["prediction"],
        "Good_Probability": result["probabilities"]["Good"],
        "Poor_Probability": result["probabilities"]["Poor"],
        "Standard_Probability": result["probabilities"]["Standard"]
    })

deployment_results_df = pd.DataFrame(deployment_results)

display(deployment_results_df)

,Customer_ID,Actual,Predicted,Good_Probability,Poor_Probability,Standard_Probability
0,CUS_0xa66b,Standard,Standard,0.055941,0.143044,0.801014
1,CUS_0x7878,Standard,Standard,0.158245,0.117991,0.723763
2,CUS_0x3702,Standard,Standard,0.009663,0.020902,0.969435


In [24]:
# Test deployment across all three credit-score classes

classes_to_test = ["Good", "Poor", "Standard"]

multi_class_results = []

for target_class in classes_to_test:
    # Find the first test customer with this actual class
    matching_index = y_test[y_test == target_class].index[0]
    
    customer = test_df.loc[[matching_index]]
    
    result = predict_credit_score(customer)
    
    multi_class_results.append({
        "Customer_ID": customer["Customer_ID"].iloc[0],
        "Actual": target_class,
        "Predicted": result["prediction"],
        "Good_Probability": result["probabilities"]["Good"],
        "Poor_Probability": result["probabilities"]["Poor"],
        "Standard_Probability": result["probabilities"]["Standard"]
    })

multi_class_results_df = pd.DataFrame(multi_class_results)

display(multi_class_results_df)

,Customer_ID,Actual,Predicted,Good_Probability,Poor_Probability,Standard_Probability
0,CUS_0x3553,Good,Good,0.669816,0.038684,0.291500
1,CUS_0x42ac,Poor,Poor,0.008099,0.841515,0.150385
2,CUS_0xa66b,Standard,Standard,0.055941,0.143044,0.801014


In [25]:
# Prepare a real customer for API testing

api_customer = test_df[test_df["Customer_ID"] == "CUS_0xa66b"].iloc[0]

api_customer_json = api_customer.to_dict()

print(api_customer_json)

{'ID': '0x167a', 'Customer_ID': 'CUS_0xa66b', 'Month': 'January', 'Name': nan, 'Age': 40.0, 'SSN': '221-30-8554', 'Occupation': 'Teacher', 'Annual_Income': 33751.27, 'Monthly_Inhand_Salary': 2948.605833333333, 'Num_Bank_Accounts': 5, 'Num_Credit_Card': 5, 'Interest_Rate': 20, 'Num_of_Loan': 3, 'Delay_from_due_date': 16, 'Num_of_Delayed_Payment': 20.0, 'Changed_Credit_Limit': 11.0, 'Num_Credit_Inquiries': 4.0, 'Credit_Mix': 'Standard', 'Outstanding_Debt': 1328.93, 'Credit_Utilization_Ratio': 37.08907564543987, 'Payment_of_Min_Amount': 'NM', 'Total_EMI_per_month': 65.00817428651536, 'Amount_invested_monthly': 117.30669710658556, 'Payment_Behaviour': 'High_spent_Medium_value_payments', 'Monthly_Balance': 362.5457119402324, 'Credit_Score': 'Standard', 'Credit_History_Months': 230.0, 'Num_Loan_Types': 3, 'Has_auto_loan': 1, 'Has_credit_builder_loan': 1, 'Has_personal_loan': 1, 'Has_home_equity_loan': 0, 'Has_mortgage_loan': 0, 'Has_student_loan': 0, 'Has_debt_consolidation_loan': 0, 'Has_pa

In [26]:
import json

print(json.dumps(api_customer_json, indent=2, default=str))

{
  "ID": "0x167a",
  "Customer_ID": "CUS_0xa66b",
  "Month": "January",
  "Name": NaN,
  "Age": 40.0,
  "SSN": "221-30-8554",
  "Occupation": "Teacher",
  "Annual_Income": 33751.27,
  "Monthly_Inhand_Salary": 2948.605833333333,
  "Num_Bank_Accounts": 5,
  "Num_Credit_Card": 5,
  "Interest_Rate": 20,
  "Num_of_Loan": 3,
  "Delay_from_due_date": 16,
  "Num_of_Delayed_Payment": 20.0,
  "Changed_Credit_Limit": 11.0,
  "Num_Credit_Inquiries": 4.0,
  "Credit_Mix": "Standard",
  "Outstanding_Debt": 1328.93,
  "Credit_Utilization_Ratio": 37.08907564543987,
  "Payment_of_Min_Amount": "NM",
  "Total_EMI_per_month": 65.00817428651536,
  "Amount_invested_monthly": 117.30669710658556,
  "Payment_Behaviour": "High_spent_Medium_value_payments",
  "Monthly_Balance": 362.5457119402324,
  "Credit_Score": "Standard",
  "Credit_History_Months": 230.0,
  "Num_Loan_Types": 3,
  "Has_auto_loan": 1,
  "Has_credit_builder_loan": 1,
  "Has_personal_loan": 1,
  "Has_home_equity_loan": 0,
  "Has_mortgage_loan": 

In [27]:
# Create clean API request data

api_customer_request = api_customer_json.copy()

# Remove the target variable
api_customer_request.pop("Credit_Score", None)

# Replace missing Name with None (valid JSON null)
if pd.isna(api_customer_request.get("Name")):
    api_customer_request["Name"] = None

print(json.dumps(api_customer_request, indent=2))

{
  "ID": "0x167a",
  "Customer_ID": "CUS_0xa66b",
  "Month": "January",
  "Name": null,
  "Age": 40.0,
  "SSN": "221-30-8554",
  "Occupation": "Teacher",
  "Annual_Income": 33751.27,
  "Monthly_Inhand_Salary": 2948.605833333333,
  "Num_Bank_Accounts": 5,
  "Num_Credit_Card": 5,
  "Interest_Rate": 20,
  "Num_of_Loan": 3,
  "Delay_from_due_date": 16,
  "Num_of_Delayed_Payment": 20.0,
  "Changed_Credit_Limit": 11.0,
  "Num_Credit_Inquiries": 4.0,
  "Credit_Mix": "Standard",
  "Outstanding_Debt": 1328.93,
  "Credit_Utilization_Ratio": 37.08907564543987,
  "Payment_of_Min_Amount": "NM",
  "Total_EMI_per_month": 65.00817428651536,
  "Amount_invested_monthly": 117.30669710658556,
  "Payment_Behaviour": "High_spent_Medium_value_payments",
  "Monthly_Balance": 362.5457119402324,
  "Credit_History_Months": 230.0,
  "Num_Loan_Types": 3,
  "Has_auto_loan": 1,
  "Has_credit_builder_loan": 1,
  "Has_personal_loan": 1,
  "Has_home_equity_loan": 0,
  "Has_mortgage_loan": 0,
  "Has_student_loan": 0,
 

In [28]:
# Identify the raw features expected by the preprocessor

raw_features = (
    list(preprocessor.transformers_[0][2])
    + list(preprocessor.transformers_[1][2])
)

print("Number of raw model features:", len(raw_features))
print("\nRaw model features:")

for feature in raw_features:
    print(feature)

Number of raw model features: 31

Raw model features:
Age
Annual_Income
Monthly_Inhand_Salary
Num_Bank_Accounts
Num_Credit_Card
Interest_Rate
Num_of_Loan
Delay_from_due_date
Num_of_Delayed_Payment
Changed_Credit_Limit
Num_Credit_Inquiries
Outstanding_Debt
Credit_Utilization_Ratio
Total_EMI_per_month
Amount_invested_monthly
Monthly_Balance
Credit_History_Months
Num_Loan_Types
Has_auto_loan
Has_credit_builder_loan
Has_personal_loan
Has_home_equity_loan
Has_mortgage_loan
Has_student_loan
Has_debt_consolidation_loan
Has_payday_loan
Month
Occupation
Credit_Mix
Payment_of_Min_Amount
Payment_Behaviour


In [29]:
# Check Package versions
import sys
import pandas
import numpy
import sklearn
import xgboost
import shap
import joblib
import fastapi
import uvicorn

print("Python:", sys.version.split()[0])
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("shap:", shap.__version__)
print("joblib:", joblib.__version__)
print("fastapi:", fastapi.__version__)
print("uvicorn:", uvicorn.__version__)

Python: 3.8.5
pandas: 1.4.4
numpy: 1.19.2
scikit-learn: 1.1.3
xgboost: 1.5.0
shap: 0.40.0
joblib: 1.1.0
fastapi: 0.124.4
uvicorn: 0.33.0


In [30]:
import pydantic
import typing_extensions

print("FastAPI:", fastapi.__version__)
print("Pydantic:", pydantic.__version__)
print("typing_extensions:", typing_extensions.__file__)

FastAPI: 0.124.4
Pydantic: 2.10.6
typing_extensions: C:\Users\Lum-neh Angela\anaconda3\lib\site-packages\typing_extensions.py


In [31]:
from pathlib import Path

requirements_path = PROJECT_ROOT / "requirements.txt"

print("Requirements file:", requirements_path)
print("Exists:", requirements_path.exists())

if requirements_path.exists():
    print("\nContents:\n")
    print(requirements_path.read_text())

Requirements file: C:\Users\Lum-neh Angela\Documents\Data\CreditScoring_Kaggle\requirements.txt
Exists: True

Contents:

pandas==1.4.4
numpy==1.19.2
scikit-learn==1.1.3
xgboost==1.5.0
shap==0.40.0
joblib==1.1.0
fastapi==0.124.4
uvicorn==0.33.0
pydantic==2.10.6
typing-extensions==4.13.2


In [32]:
# Test Monitoring module
from src.monitoring import CreditScoreMonitor

print("CreditScoreMonitor imported successfully.")

CreditScoreMonitor imported successfully.
